In [1]:
# ============================================================
# INSTALL DEPENDENCIES
# ============================================================

!pip install -q torch torchaudio librosa soundfile numpy pandas scikit-learn tqdm matplotlib seaborn \
    huggingface_hub "datasets[audio]" transformers pydub rapidfuzz faster-whisper

!apt-get -qq update
!apt-get -qq install -y ffmpeg


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
import os
import re
import random
import shutil
import tempfile
import subprocess
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)

from tqdm.auto import tqdm
from pydub import AudioSegment

# ==========================================
# SETTINGS
# ==========================================

SAMPLE_RATE = 16000
DURATION = 4
TARGET_LENGTH = SAMPLE_RATE * DURATION

BATCH_SIZE = 16
EPOCHS = 12
LEARNING_RATE = 1e-3

WAV2VEC_MODEL_NAME = "facebook/wav2vec2-base"   # frozen pretrained speech backbone

# Starting point only — the real threshold gets recalibrated later in this notebook.
THRESHOLD = 0.50

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print("Device:", DEVICE)


Device: cuda


In [3]:
# ==========================================
# PROJECT FOLDERS
# ==========================================

BASE_DIR = Path("/content/VoiceShieldBharat")

DATA_DIR = BASE_DIR / "data"
EXTRA_FAKE_DIR = DATA_DIR / "extra_fake_sources"   # <-- put ElevenLabs / other AI clips here
MODEL_DIR = BASE_DIR / "models"
REPORT_DIR = BASE_DIR / "reports"

for folder in [DATA_DIR, EXTRA_FAKE_DIR, MODEL_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", BASE_DIR)
print()
print("Drop additional AI-generated clips (ElevenLabs, Bark, XTTS, gTTS, edge-tts, PlayHT, etc.)")
print("into this folder before training, ideally in a few per-generator subfolders:")
print(EXTRA_FAKE_DIR)


Project folder: /content/VoiceShieldBharat

Drop additional AI-generated clips (ElevenLabs, Bark, XTTS, gTTS, edge-tts, PlayHT, etc.)
into this folder before training, ideally in a few per-generator subfolders:
/content/VoiceShieldBharat/data/extra_fake_sources


In [4]:
# ==========================================
# LOAD BASE VOICEGUARD DATASET
# ==========================================

DATASET_DIR = Path("/content/voiceguard_dataset")
ZIP_PATH = DATASET_DIR / "voiceguard-competition.zip"
EXTRACT_DIR = DATASET_DIR / "extracted"

DATASET_DIR.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    print("Downloading voiceguard-competition dataset from Hugging Face Hub...")
    from huggingface_hub import hf_hub_download
    hf_hub_download(
        repo_id="fassabilf/voiceguard-competition",
        filename="voiceguard-competition.zip",
        repo_type="dataset",
        local_dir=DATASET_DIR,
        local_dir_use_symlinks=False
    )
    print("Download complete.")

if not EXTRACT_DIR.exists():
    import zipfile
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("Extraction complete.")
else:
    print("Dataset already extracted.")

all_wav_files = list(EXTRACT_DIR.rglob("*.wav"))
base_real = [p for p in all_wav_files if "/real/" in p.as_posix()]
base_fake = [p for p in all_wav_files if "/fake/" in p.as_posix()]

print("\nBase REAL files:", len(base_real))
print("Base FAKE files:", len(base_fake))


Dataset already extracted.

Base REAL files: 2874
Base FAKE files: 2874


In [5]:
# ==========================================
# BRING IN YOUR "UNSEEN GENERATOR" FAKE SAMPLES
# (ElevenLabs / any other TTS you have on hand)
# ==========================================

extra_fake_files = [
    p for p in EXTRA_FAKE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".wav", ".mp3", ".m4a", ".ogg", ".flac"}
]

print("Extra (unseen-generator) fake files found:", len(extra_fake_files))
for p in extra_fake_files[:10]:
    print(" -", p)

if len(extra_fake_files) == 0:
    print()
    print("WARNING: no extra fake samples found. The model will only ever see one generator's")
    print("artifacts and the ElevenLabs-style generalization problem will persist. Add clips to:")
    print(EXTRA_FAKE_DIR)


Extra (unseen-generator) fake files found: 0

artifacts and the ElevenLabs-style generalization problem will persist. Add clips to:
/content/VoiceShieldBharat/data/extra_fake_sources


In [6]:
# ==========================================
# BUILD COMBINED DATAFRAME WITH SOURCE TAGS
# 0 = REAL, 1 = FAKE
# source is tracked so we can report per-source metrics later
# ==========================================

files = (
    [str(p) for p in base_real] +
    [str(p) for p in base_fake] +
    [str(p) for p in extra_fake_files]
)

labels = (
    [0] * len(base_real) +
    [1] * len(base_fake) +
    [1] * len(extra_fake_files)
)

sources = (
    ["voiceguard_real"] * len(base_real) +
    ["voiceguard_fake"] * len(base_fake) +
    ["extra_generators"] * len(extra_fake_files)
)

audio_df = pd.DataFrame({"path": files, "label": labels, "source": sources})

print("Total samples:", len(audio_df))
print("\nClass distribution:")
print(audio_df["label"].map({0: "REAL", 1: "FAKE"}).value_counts())
print("\nBy source:")
print(audio_df["source"].value_counts())


Total samples: 5748

Class distribution:
label
REAL    2874
FAKE    2874
Name: count, dtype: int64

By source:
source
voiceguard_real    2874
voiceguard_fake    2874
Name: count, dtype: int64


In [7]:
# ==========================================
# AUDIO LOADING + AUGMENTATION
# Silence trimming and loudness normalization are applied identically at
# train time and inference time so the model can't key on artifacts that
# differ only because files came from different pipelines.
# ==========================================

def trim_silence(audio, top_db=25):
    trimmed, _ = librosa.effects.trim(audio, top_db=top_db)
    return trimmed if len(trimmed) > 0 else audio


def add_noise(audio, noise_level=0.005):
    noise = np.random.randn(len(audio)).astype(np.float32)
    return audio + noise_level * noise


def random_gain(audio):
    gain = np.random.uniform(0.7, 1.3)
    return audio * gain


def random_codec_reencode(audio, sr):
    """
    Round-trip the audio through a lossy MP3 codec at a random bitrate.
    This stops the model from latching onto a particular file's codec
    signature as a shortcut for 'fake', since real-world incoming audio
    (phone calls, downloaded clips, different apps) all carry different
    codec fingerprints.
    """
    try:
        with tempfile.TemporaryDirectory() as tmp:
            wav_path = os.path.join(tmp, "in.wav")
            mp3_path = os.path.join(tmp, "out.mp3")
            sf.write(wav_path, audio, sr)
            bitrate = random.choice(["24k", "32k", "48k", "64k", "96k"])
            AudioSegment.from_wav(wav_path).export(mp3_path, format="mp3", bitrate=bitrate)
            y, _ = librosa.load(mp3_path, sr=sr, mono=True)
            return y
    except Exception:
        return audio


def load_and_fix_audio(path, augment=False):
    audio, sr = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    audio = np.nan_to_num(audio)
    audio = trim_silence(audio)

    if augment:
        if random.random() < 0.5:
            audio = add_noise(audio)
        if random.random() < 0.5:
            audio = random_gain(audio)
        if random.random() < 0.35:
            audio = random_codec_reencode(audio, SAMPLE_RATE)

    max_val = np.max(np.abs(audio))
    if max_val > 0:
        audio = audio / max_val

    if len(audio) < TARGET_LENGTH:
        audio = np.pad(audio, (0, TARGET_LENGTH - len(audio)))
    else:
        if augment and len(audio) > TARGET_LENGTH:
            start = random.randint(0, len(audio) - TARGET_LENGTH)
        else:
            start = 0
        audio = audio[start:start + TARGET_LENGTH]

    return audio.astype(np.float32)


print("Audio loading + augmentation ready.")


Audio loading + augmentation ready.


In [8]:
# ==========================================
# FROZEN WAV2VEC2 FEATURE EXTRACTOR
# A pretrained speech backbone generalizes far better across unseen TTS
# engines than a small CNN trained on one dataset's mel-spectrograms,
# because it was pretrained on a wide, generic speech distribution rather
# than memorizing one generator's artifacts. We freeze it and only train
# a small classifier head, which also keeps training fast and avoids
# overfitting on our fairly small hackathon dataset.
# ==========================================

from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model

wav2vec_processor = Wav2Vec2FeatureExtractor.from_pretrained(WAV2VEC_MODEL_NAME)
wav2vec_backbone = Wav2Vec2Model.from_pretrained(WAV2VEC_MODEL_NAME).to(DEVICE)
wav2vec_backbone.eval()

for p in wav2vec_backbone.parameters():
    p.requires_grad = False

EMBEDDING_DIM = wav2vec_backbone.config.hidden_size * 2  # mean + std pooling


@torch.no_grad()
def extract_embedding(audio_np):
    inputs = wav2vec_processor(
        audio_np, sampling_rate=SAMPLE_RATE, return_tensors="pt"
    )
    input_values = inputs.input_values.to(DEVICE)
    hidden = wav2vec_backbone(input_values).last_hidden_state  # [1, T, H]

    mean_pool = hidden.mean(dim=1)
    std_pool = hidden.std(dim=1)
    embedding = torch.cat([mean_pool, std_pool], dim=1)  # statistics pooling

    return embedding.squeeze(0).cpu()


print("Wav2Vec2 backbone loaded and frozen. Embedding dim:", EMBEDDING_DIM)


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Wav2Vec2 backbone loaded and frozen. Embedding dim: 1536


In [9]:
# ==========================================
# TRAIN / VALIDATION / TEST SPLIT
# Stratified by label + source together so extra_fake (unseen-generator)
# samples show up in every split, not just training.
# ==========================================

strat_key = [f"{l}_{s}" for l, s in zip(audio_df["label"], audio_df["source"])]
key_counts = Counter(strat_key)
safe_stratify = strat_key if min(key_counts.values()) >= 2 else None

if safe_stratify is None:
    print("Some source/label groups are too small to stratify by source; falling back to label-only stratification.")
    safe_stratify = list(audio_df["label"])

train_df, temp_df = train_test_split(
    audio_df, test_size=0.30, random_state=42, stratify=safe_stratify
)

temp_strat = temp_df["label"] if safe_stratify is list(audio_df["label"]) else None
try:
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, random_state=42,
        stratify=[f"{l}_{s}" for l, s in zip(temp_df['label'], temp_df['source'])]
    )
except ValueError:
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"]
    )

print("TRAIN:", len(train_df), " VAL:", len(val_df), " TEST:", len(test_df))
print("\nTest set by source:")
print(test_df["source"].value_counts())


TRAIN: 4023  VAL: 862  TEST: 863

Test set by source:
source
voiceguard_real    432
voiceguard_fake    431
Name: count, dtype: int64


In [10]:
# ==========================================
# PYTORCH DATASET
# ==========================================

class VoiceDataset(Dataset):
    def __init__(self, df, augment=False):
        self.paths = df["path"].tolist()
        self.labels = df["label"].tolist()
        self.augment = augment

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        audio = load_and_fix_audio(self.paths[idx], augment=self.augment)
        embedding = extract_embedding(audio)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return embedding, label


train_dataset = VoiceDataset(train_df, augment=True)
val_dataset = VoiceDataset(val_df, augment=False)
test_dataset = VoiceDataset(test_df, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Test batches :", len(test_loader))


Train batches: 252
Val batches  : 54
Test batches : 54


In [11]:
# ============================================================
# ADD YOUR OWN REAL AUDIO + ELEVENLABS FAKE AUDIO
# (upload your real recordings into USER_REAL_DIR first,
#  and your ElevenLabs clips into ELEVENLABS_DIR)
# ============================================================

USER_REAL_DIR = DATA_DIR / "user_real"          # already created in Cell 2
ELEVENLABS_DIR = Path("/content/elevenlabs_samples")
ELEVENLABS_DIR.mkdir(parents=True, exist_ok=True)

def collect_audio(folder):
    exts = ("*.wav", "*.mp3", "*.m4a", "*.ogg")
    files = []
    for ext in exts:
        files += list(folder.rglob(ext))
    return files

user_real_files  = collect_audio(USER_REAL_DIR)
elevenlabs_files = collect_audio(ELEVENLABS_DIR)

print("Your own REAL samples found:", len(user_real_files))
print("ElevenLabs FAKE samples found:", len(elevenlabs_files))

extra_df = pd.DataFrame({
    "path": [str(p) for p in user_real_files] + [str(p) for p in elevenlabs_files],
    "label": [0] * len(user_real_files) + [1] * len(elevenlabs_files)
})

audio_df = pd.concat([audio_df, extra_df], ignore_index=True)

# rebuild all_files / all_labels since the split cell (Cell 15) uses these directly
all_files  = audio_df["path"].tolist()
all_labels = audio_df["label"].tolist()

print("\nUpdated class distribution:")
print(audio_df["label"].map({0: "REAL", 1: "FAKE"}).value_counts())

Your own REAL samples found: 0
ElevenLabs FAKE samples found: 0

Updated class distribution:
label
REAL    2874
FAKE    2874
Name: count, dtype: int64


In [14]:
# run once per folder — a file picker will pop up
from google.colab import files

# Ensure the directory for real audio clips exists
USER_REAL_DIR.mkdir(parents=True, exist_ok=True)
print("Upload YOUR real audio clips:")
uploaded = files.upload()
for name, data in uploaded.items():
    (USER_REAL_DIR / name).write_bytes(data)

# Ensure the directory for ElevenLabs fake clips exists
ELEVENLABS_DIR.mkdir(parents=True, exist_ok=True)
print("Upload ElevenLabs fake clips:")
uploaded = files.upload()
for name, data in uploaded.items():
    (ELEVENLABS_DIR / name).write_bytes(data)

Upload YOUR real audio clips:


Saving WhatsApp Audio 2026-09-08 at 9.32.54 PM (1).mpeg to WhatsApp Audio 2026-09-08 at 9.32.54 PM (1) (2).mpeg
Saving WhatsApp Audio 2026-09-08 at 9.32.54 PM (2).mpeg to WhatsApp Audio 2026-09-08 at 9.32.54 PM (2) (2).mpeg
Saving WhatsApp Audio 2026-09-08 at 9.32.54 PM.mpeg to WhatsApp Audio 2026-09-08 at 9.32.54 PM (4).mpeg
Saving WhatsApp Audio 2026-09-08 at 9.32.55 PM (1).mpeg to WhatsApp Audio 2026-09-08 at 9.32.55 PM (1) (2).mpeg
Saving WhatsApp Audio 2026-09-08 at 9.32.55 PM (2).mpeg to WhatsApp Audio 2026-09-08 at 9.32.55 PM (2) (2).mpeg
Saving WhatsApp Audio 2026-09-08 at 9.32.55 PM.mpeg to WhatsApp Audio 2026-09-08 at 9.32.55 PM (4).mpeg
Saving WhatsApp Audio 2026-09-08 at 9.33.26 PM (1).mpeg to WhatsApp Audio 2026-09-08 at 9.33.26 PM (1) (2).mpeg
Saving WhatsApp Audio 2026-09-08 at 9.33.26 PM.mp4 to WhatsApp Audio 2026-09-08 at 9.33.26 PM (2).mp4
Saving WhatsApp Audio 2026-09-08 at 9.33.26 PM.mpeg to WhatsApp Audio 2026-09-08 at 9.33.26 PM (3).mpeg
Saving WhatsApp Audio 2026

Saving ElevenLabs_2026-09-08T20_03_33_Roger - Laid-Back, Casual, Resonant_pre_sp100_s50_sb75_se0_b_e2.mp3 to ElevenLabs_2026-09-08T20_03_33_Roger - Laid-Back, Casual, Resonant_pre_sp100_s50_sb75_se0_b_e2.mp3
Saving ElevenLabs_2026-09-08T20_12_41_Roger - Laid-Back, Casual, Resonant_pre_sp100_s50_sb75_se0_b_m2.mp3 to ElevenLabs_2026-09-08T20_12_41_Roger - Laid-Back, Casual, Resonant_pre_sp100_s50_sb75_se0_b_m2.mp3
Saving ElevenLabs_2026-09-09T05_50_06_Roger - Laid-Back, Casual, Resonant_pre_sp100_s50_sb75_se0_b_m2.mp3 to ElevenLabs_2026-09-09T05_50_06_Roger - Laid-Back, Casual, Resonant_pre_sp100_s50_sb75_se0_b_m2.mp3
Saving ElevenLabs_2026-09-09T05_51_57_Roger - Laid-Back, Casual, Resonant_pre_sp100_s50_sb75_se0_b_m2 (1).mp3 to ElevenLabs_2026-09-09T05_51_57_Roger - Laid-Back, Casual, Resonant_pre_sp100_s50_sb75_se0_b_m2 (1).mp3
Saving ElevenLabs_2026-09-09T05_51_57_Roger - Laid-Back, Casual, Resonant_pre_sp100_s50_sb75_se0_b_m2.mp3 to ElevenLabs_2026-09-09T05_51_57_Roger - Laid-Back, C

In [15]:
# ==========================================
# CLASSIFIER HEAD (trained) on top of frozen wav2vec2 embeddings
# ==========================================

class SpoofClassifierHead(nn.Module):
    def __init__(self, input_dim=EMBEDDING_DIM, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(hidden, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.net(x)


model = SpoofClassifierHead().to(DEVICE)
print(model)


SpoofClassifierHead(
  (net): Sequential(
    (0): Linear(in_features=1536, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.35, inplace=False)
    (3): Linear(in_features=256, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=2, bias=True)
  )
)


In [16]:
# ==========================================
# LOSS (class-weighted) + OPTIMIZER
# ==========================================

label_counts = Counter(train_df["label"])
total = sum(label_counts.values())
class_weights = torch.tensor(
    [total / (2 * label_counts[0]), total / (2 * label_counts[1])],
    dtype=torch.float32
).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

MODEL_PATH = MODEL_DIR / "voiceshield_head_best.pth"
best_val_loss = float("inf")
history = {"train_loss": [], "val_loss": [], "val_accuracy": []}

print("Class weights (real, fake):", class_weights.tolist())
print("Model path:", MODEL_PATH)


Class weights (real, fake): [1.000248670578003, 0.9997515082359314]
Model path: /content/VoiceShieldBharat/models/voiceshield_head_best.pth


In [ ]:
# ==========================================
# TRAIN
# ==========================================

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    for X, y in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}"):
        X, y = X.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        output = model(X)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()
    val_loss, correct, total_n = 0.0, 0, 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            output = model(X)
            loss = criterion(output, y)
            val_loss += loss.item()
            preds = torch.argmax(output, dim=1)
            correct += (preds == y).sum().item()
            total_n += y.size(0)

    val_loss /= len(val_loader)
    val_accuracy = correct / total_n
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_PATH)
        print("  -> best model saved")

print("\nTraining complete. Best model:", MODEL_PATH)


Epoch 1/12:   0%|          | 0/252 [00:00<?, ?it/s]

Epoch 1/12 | Train Loss: 0.3893 | Val Loss: 0.0854 | Val Acc: 0.9722
  -> best model saved


Epoch 2/12:   0%|          | 0/252 [00:00<?, ?it/s]

In [ ]:
# ==========================================
# TRAINING CURVES
# ==========================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))


axes[0].plot(history["train_loss"], label="Train Loss")
axes[0].plot(history["val_loss"], label="Val Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend()
axes[0].set_title("Loss")

axes[1].plot(history["val_accuracy"], label="Val Accuracy", color="green")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy"); axes[1].legend()
axes[1].set_title("Validation Accuracy")

plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# LOAD BEST MODEL
# ==========================================

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print("Best model loaded.")


In [ ]:
# ==========================================
# THRESHOLD CALIBRATION ON VALIDATION SET
# Instead of a hardcoded 0.70, sweep thresholds and pick the one that
# maximizes F1 on held-out validation data (covers both the base dataset
# and any extra-generator samples you added).
# ==========================================

val_probs, val_labels_true = [], []

model.eval()
with torch.no_grad():
    for X, y in val_loader:
        X = X.to(DEVICE)
        probs = torch.softmax(model(X), dim=1)[:, 1]
        val_probs.extend(probs.cpu().numpy())
        val_labels_true.extend(y.numpy())

val_probs = np.array(val_probs)
val_labels_true = np.array(val_labels_true)

best_threshold, best_f1 = 0.5, -1
for t in np.arange(0.20, 0.85, 0.02):
    preds = (val_probs >= t).astype(int)
    f1 = f1_score(val_labels_true, preds, zero_division=0)
    if f1 > best_f1:
        best_f1, best_threshold = f1, t

THRESHOLD = float(best_threshold)
print(f"Calibrated THRESHOLD = {THRESHOLD:.2f}  (validation F1 = {best_f1:.4f})")


In [ ]:
# ==========================================
# TEST SET EVALUATION — OVERALL AND PER SOURCE
# The per-source breakdown is the key evidence: it shows whether the
# model still generalizes to the "extra_generators" (e.g. ElevenLabs)
# samples it was partially trained/validated on, versus samples it has
# genuinely never seen (swap in a fresh clip in the manual test cell below).
# ==========================================

test_paths = test_df["path"].tolist()
test_labels_list = test_df["label"].tolist()
test_sources = test_df["source"].tolist()

all_probs, all_preds = [], []

model.eval()
with torch.no_grad():
    for path in tqdm(test_paths, desc="Testing"):
        audio = load_and_fix_audio(path, augment=False)
        emb = extract_embedding(audio).unsqueeze(0).to(DEVICE)
        prob = torch.softmax(model(emb), dim=1)[0, 1].item()
        all_probs.append(prob)
        all_preds.append(int(prob >= THRESHOLD))

results_df = pd.DataFrame({
    "path": test_paths,
    "source": test_sources,
    "actual": test_labels_list,
    "predicted": all_preds,
    "fake_probability": all_probs
})

print("=" * 60)
print("OVERALL TEST METRICS")
print("=" * 60)
print(f"Accuracy  : {accuracy_score(test_labels_list, all_preds):.4f}")
print(f"Precision : {precision_score(test_labels_list, all_preds, zero_division=0):.4f}")
print(f"Recall    : {recall_score(test_labels_list, all_preds, zero_division=0):.4f}")
print(f"F1 Score  : {f1_score(test_labels_list, all_preds, zero_division=0):.4f}")
try:
    print(f"ROC-AUC   : {roc_auc_score(test_labels_list, all_probs):.4f}")
except ValueError:
    pass

print("\n" + "=" * 60)
print("PER-SOURCE BREAKDOWN")
print("=" * 60)
for src, group in results_df.groupby("source"):
    acc = accuracy_score(group["actual"], group["predicted"])
    print(f"{src:20s} n={len(group):4d}  accuracy={acc:.4f}")


In [ ]:
# ==========================================
# CONFUSION MATRIX
# ==========================================

cm = confusion_matrix(test_labels_list, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["REAL", "FAKE"])
disp.plot()
plt.title("VoiceShield Bharat — Test Confusion Matrix")
plt.show()


In [ ]:
# ==========================================
# SAVE METRICS + PREDICTIONS
# ==========================================

report = {
    "accuracy": accuracy_score(test_labels_list, all_preds),
    "precision": precision_score(test_labels_list, all_preds, zero_division=0),
    "recall": recall_score(test_labels_list, all_preds, zero_division=0),
    "f1_score": f1_score(test_labels_list, all_preds, zero_division=0),
    "threshold": THRESHOLD
}
pd.DataFrame([report]).to_csv(REPORT_DIR / "voiceshield_metrics.csv", index=False)
results_df.to_csv(REPORT_DIR / "test_predictions.csv", index=False)

print("Saved metrics and predictions to:", REPORT_DIR)
print(pd.DataFrame([report]).to_string(index=False))


In [ ]:
# ==========================================
# UNIVERSAL AUDIO CONVERTER (any format -> 16kHz mono wav)
# ==========================================

def convert_to_wav(input_file, output_file=None):
    input_file = str(input_file)
    if output_file is None:
        output_file = str(Path(tempfile.gettempdir()) / (Path(input_file).stem + "_conv.wav"))

    command = ["ffmpeg", "-y", "-i", input_file, "-vn", "-ac", "1", "-ar", str(SAMPLE_RATE),
               "-sample_fmt", "s16", output_file]
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Audio conversion failed.")

    return output_file


def analyze_incoming_audio(audio_file):
    """Full inference path: any format -> fixed/trimmed audio -> wav2vec2 embedding -> classifier."""
    wav_file = convert_to_wav(audio_file)
    audio = load_and_fix_audio(wav_file, augment=False)

    with torch.no_grad():
        emb = extract_embedding(audio).unsqueeze(0).to(DEVICE)
        probabilities = torch.softmax(model(emb), dim=1)[0]

    real_probability = float(probabilities[0].cpu())
    fake_probability = float(probabilities[1].cpu())
    decision = "SUSPICIOUS" if fake_probability >= THRESHOLD else "GENUINE"

    return {
        "real_probability": real_probability,
        "fake_probability": fake_probability,
        "decision": decision
    }


print("Detector ready. Active threshold:", THRESHOLD)


In [ ]:
from IPython.display import display, HTML


def show_call_status(result):
    fake = result["fake_probability"] * 100
    real = result["real_probability"] * 100

    if result["decision"] == "GENUINE":
        display(HTML(f"""
        <div style="background:#d4edda;border:3px solid #28a745;padding:25px;border-radius:15px;text-align:center;font-family:Arial;">
            <h1 style="color:#155724;">🟢 VOICE APPEARS GENUINE</h1>
            <h2>Real Probability: {real:.2f}%</h2>
            <h2>Fake Probability: {fake:.2f}%</h2>
            <h2 style="color:#155724;">CALL CAN CONTINUE</h2>
        </div>
        """))
    else:
        display(HTML(f"""
        <div style="background:#f8d7da;border:4px solid #dc3545;padding:25px;border-radius:15px;text-align:center;font-family:Arial;">
            <h1 style="color:#721c24;">🔴 AI / SPOOF VOICE SUSPECTED</h1>
            <h2>Fake Probability: {fake:.2f}%</h2>
            <h2>Real Probability: {real:.2f}%</h2>
            <h2 style="color:#721c24;">⚠️ CALL TEMPORARILY BLOCKED</h2>
            <p>Liveness verification required.</p>
        </div>
        """))


In [ ]:
# ==========================================
# QUICK MANUAL TEST — upload an ElevenLabs (or any) clip to sanity check
# ==========================================

from google.colab import files

uploaded = files.upload()
uploaded_file = list(uploaded.keys())[0]

print("Testing:", uploaded_file)
result = analyze_incoming_audio(uploaded_file)
print(result)
show_call_status(result)


In [ ]:
!pip install -q faster-whisper

from faster_whisper import WhisperModel

whisper_model = WhisperModel("base", device="cpu", compute_type="int8")
print("Multilingual Whisper loaded.")


In [ ]:
CHALLENGES = [
    {
        "display": "भारत सुरक्षित है",
        "language": "Hindi",
        "acceptable": ["भारत सुरक्षित है", "bharat surakshit hai", "bharat surakshit h"]
    },
    {
        "display": "Bharat is safe",
        "language": "English",
        "acceptable": ["bharat is safe", "bharat safe hai", "india is safe"]
    },
    {
        "display": "Bharat safe hai",
        "language": "Hinglish",
        "acceptable": ["bharat safe hai", "bharat surakshit hai", "bharat safe h"]
    }
]


def get_random_challenge():
    return random.choice(CHALLENGES)


In [ ]:
from IPython.display import Javascript
from google.colab import output
from base64 import b64decode

RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const blobToBase64 = blob => new Promise(resolve => {
    const reader = new FileReader();
    reader.onloadend = () => resolve(reader.result);
    reader.readAsDataURL(blob);
});

async function recordAudio(seconds) {
    const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
    const recorder = new MediaRecorder(stream);
    let chunks = [];
    recorder.ondataavailable = event => { if (event.data.size > 0) chunks.push(event.data); };
    recorder.start();
    await sleep(seconds * 1000);
    recorder.stop();
    await new Promise(resolve => recorder.onstop = resolve);
    const blob = new Blob(chunks, { type: 'audio/webm' });
    const base64 = await blobToBase64(blob);
    return base64;
}
"""


def record_microphone(seconds=5):
    display(Javascript(RECORD_JS))
    from google.colab.output import eval_js
    data = eval_js(f"recordAudio({seconds})")
    binary = b64decode(data.split(',')[1])
    webm_path = "/content/liveness_response.webm"
    with open(webm_path, "wb") as f:
        f.write(binary)
    return webm_path


def prepare_liveness_audio(webm_file):
    wav_file = "/content/liveness_response.wav"
    convert_to_wav(webm_file, wav_file)
    return wav_file


def transcribe_audio(audio_file):
    segments, _ = whisper_model.transcribe(audio_file, language=None, beam_size=5)
    return " ".join(segment.text for segment in segments).strip()


def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r"[^\w\s\u0900-\u097F]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text


from rapidfuzz.fuzz import ratio


def challenge_matches(transcript, challenge, threshold=65):
    transcript = normalize_text(transcript)
    best_score, best_match = 0, None
    for expected in challenge["acceptable"]:
        expected = normalize_text(expected)
        score = ratio(transcript, expected)
        if score > best_score:
            best_score, best_match = score, expected
    return best_score >= threshold, best_score, best_match


def check_liveness_response(audio_file):
    result = analyze_incoming_audio(audio_file)
    transcript = transcribe_audio(audio_file)
    return {
        "fake_probability": result["fake_probability"],
        "real_probability": result["real_probability"],
        "transcript": transcript
    }


In [ ]:
def run_liveness_challenge():
    challenge = get_random_challenge()

    display(HTML(f"""
    <div style="background:#fff3cd;border:3px solid #ffc107;padding:25px;border-radius:15px;text-align:center;font-family:Arial;">
        <h1>⚠️ LIVENESS VERIFICATION</h1>
        <h2>Please say:</h2>
        <h1>"{challenge['display']}"</h1>
        <p>Language: {challenge['language']}</p>
    </div>
    """))

    webm_file = record_microphone(seconds=5)
    wav_file = prepare_liveness_audio(webm_file)
    response = check_liveness_response(wav_file)

    transcript = response["transcript"]
    fake_probability = response["fake_probability"]

    matched, score, expected = challenge_matches(transcript, challenge)

    print("\nTranscript:", transcript)
    print(f"Challenge Match: {score:.1f}%")
    print(f"Fake Probability: {fake_probability * 100:.2f}%")

    if matched and fake_probability < THRESHOLD:
        display(HTML("""
        <div style="background:#d4edda;border:4px solid #28a745;padding:30px;border-radius:15px;text-align:center;font-family:Arial;">
            <h1 style="color:#155724;">🟢 LIVENESS VERIFIED</h1>
            <h2>Human response detected</h2>
            <h2>Challenge matched</h2>
            <h2>CALL ALLOWED</h2>
        </div>
        """))
        return "ALLOW"
    else:
        display(HTML("""
        <div style="background:#f8d7da;border:4px solid #dc3545;padding:30px;border-radius:15px;text-align:center;font-family:Arial;">
            <h1 style="color:#721c24;">🔴 LIVENESS FAILED</h1>
            <h2>Suspicious response detected</h2>
            <h2>⚠️ CALL BLOCKED</h2>
        </div>
        """))
        return "BLOCK"


In [ ]:
def voiceshield_demo(audio_file):
    display(HTML("""
    <div style="background:#e8f0fe;padding:20px;border-radius:15px;text-align:center;">
        <h1>📞 INCOMING CALL DETECTED</h1>
        <p>VoiceShield Bharat is analyzing the audio...</p>
    </div>
    """))

    result = analyze_incoming_audio(audio_file)
    show_call_status(result)

    if result["decision"] == "GENUINE":
        return {
            "status": "ALLOW",
            "reason": "Voice appears genuine",
            "fake_probability": result["fake_probability"]
        }

    display(HTML("""
    <div style="background:#fff3cd;border:3px solid #ffc107;padding:20px;border-radius:15px;text-align:center;">
        <h2>⚠️ Additional verification required</h2>
        <p>VoiceShield has detected a potentially AI-generated voice.</p>
    </div>
    """))

    final_status = run_liveness_challenge()

    return {
        "status": final_status,
        "initial_fake_probability": result["fake_probability"]
    }


In [ ]:
# ==========================================
# FULL END-TO-END DEMO
# ==========================================

from google.colab import files

uploaded = files.upload()
incoming_audio = list(uploaded.keys())[0]

final_result = voiceshield_demo(incoming_audio)

print("\nFINAL SYSTEM DECISION:")
print(final_result)
